# 메뉴 크롤링

## 1. 메가커피

In [165]:
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd
from rapidfuzz import fuzz

print("라이브러리 설치")

라이브러리 설치


In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=service, options=options)

BASE_URL = "https://www.mega-mgccoffee.com/menu/?menu_category1=1&menu_category2=1"

CATEGORY_MAP = {
    '1': '커피',
    '2': '티',
    '3': '에이드&주스',
    '4': '스무디&프라페',
    '5': '디카페인',
    '6': '음료',
}
COFFEE_CATEGORIES = {'1', '5'}

all_items = []

def parse_page(soup, category, category_name):
    items = []
    menu_list = soup.find('ul', id='menu_list')
    if not menu_list:
        return items
    for li in menu_list.find_all('li'):
        try:
            name_tag = li.find('b')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            eng_div = li.find('div', class_='cont_text_info')
            eng_name = eng_div.find('div', class_='text').get_text(strip=True) if eng_div else ''

            label = li.find('div', class_='cont_gallery_list_label')
            temp = label.get_text(strip=True) if label else ''

            items.append({
                'brand': '메가커피',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'category': category_name,
                'is_coffee': category in COFFEE_CATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

def get_total_pages(soup):
    pages = soup.select('#board_page li a.board_page_link')
    page_numbers = [int(p.get_text(strip=True)) for p in pages if p.get_text(strip=True).isdigit()]
    return max(page_numbers) if page_numbers else 1

# 카테고리별 수집
for cat_value, cat_name in CATEGORY_MAP.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get(BASE_URL)
    WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, "menu_list")))
    time.sleep(2)

    # 체크박스 클릭
    checkbox = driver.find_element(By.CSS_SELECTOR, f'input[name="list_checkbox"][value="{cat_value}"]')
    driver.execute_script("arguments[0].click();", checkbox)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    total_pages = get_total_pages(soup)
    print(f"  총 {total_pages}페이지")

    page_items = parse_page(soup, cat_value, cat_name)
    all_items.extend(page_items)
    print(f"  1페이지: {len(page_items)}개")

    for page in range(2, total_pages + 1):
        try:
            btn = driver.find_element(By.CSS_SELECTOR, f'a.board_page_link[data-page="{page}"]')
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            page_items = parse_page(soup, cat_value, cat_name)
            all_items.extend(page_items)
            print(f"  {page}페이지: {len(page_items)}개")
        except Exception as e:
            print(f"  {page}페이지 에러: {e}")
            continue

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('mega_coffee_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 저장 → mega_coffee_menu.csv")
print(df.head(10))
print(f"\n카테고리별 수집 수:\n{df['category'].value_counts()}")


▶ 커피 수집 시작
  총 3페이지
  1페이지: 20개
  2페이지: 20개
  3페이지: 1개

▶ 티 수집 시작
  총 2페이지
  1페이지: 20개
  2페이지: 5개

▶ 에이드&주스 수집 시작
  총 1페이지
  1페이지: 12개

▶ 스무디&프라페 수집 시작
  총 1페이지
  1페이지: 20개

▶ 디카페인 수집 시작
  총 2페이지
  1페이지: 20개
  2페이지: 11개

▶ 음료 수집 시작
  총 1페이지
  1페이지: 19개

✅ 완료! 총 148개 저장 → mega_coffee_menu.csv
  brand           name                                      name_en temp  \
0  메가커피     초코젤라또 말차라떼                Choco-gelato Matcha Tea Latte  ICE   
1  메가커피     (HOT)헛개리카노                Oriental Raisin-Tea Americano  HOT   
2  메가커피     (ICE)헛개리카노                Oriental Raisin-Tea Americano  ICE   
3  메가커피        왕메가카페라떼                         BIG MEGA Caffe Latte  ICE   
4  메가커피       왕메가헛개리카노      BIG MEGA  Oriental Raisin-Tea Americano  ICE   
5  메가커피        할메가미숫커피  MEGA MGC Mix Coffee Blend with Grain Powder  ICE   
6  메가커피  라이트 바닐라 아몬드라떼                   Light vanilla almond latte  ICE   
7  메가커피           연유라떼                         Condensed Milk Latte  ICE   
8  메가커피          할메가커피

## 2. 컴포즈

In [ ]:
def parse_page(soup, cat_name):
    items = []
    boxes = soup.find_all('div', class_='itemBox')
    
    for box in boxes:
        try:
            name = box.find('div', class_='header').get_text(strip=True)
            items.append({
                'brand': '컴포즈커피',
                'name': name,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_CATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [27]:
all_items = []

for cat_name, base_url in target_categories.items():
    print(f"\n▶ {cat_name} 수집 시작")
    page = 1

    while True:
        url = f"{base_url}?page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        items = parse_page(soup, cat_name)
        if not items:
            print(f"  {page}페이지 데이터 없음 → 종료")
            break

        all_items.extend(items)
        print(f"  {page}페이지: {len(items)}개")

        # 다음 페이지 확인
        next_btn = soup.select_one('li.page-item:not(.disabled) a[aria-label="Next"]')
        if not next_btn:
            break
        page += 1
        time.sleep(1)

print(f"\n총 {len(all_items)}개 수집 완료")


▶ 시즌한정 수집 시작
  1페이지: 20개
  2페이지: 13개
  3페이지 데이터 없음 → 종료

▶ 커피 · 더치 수집 시작
  1페이지: 20개
  2페이지: 11개
  3페이지 데이터 없음 → 종료

▶ 논커피 라떼 수집 시작
  1페이지: 18개
  2페이지 데이터 없음 → 종료

▶ 프라페 · 스무디 수집 시작
  1페이지: 12개
  2페이지 데이터 없음 → 종료

▶ 밀크쉐이크 수집 시작
  1페이지: 7개
  2페이지 데이터 없음 → 종료

▶ 에이드 · 주스 수집 시작
  1페이지: 12개
  2페이지 데이터 없음 → 종료

▶ 티 수집 시작
  1페이지: 20개
  2페이지: 11개
  3페이지 데이터 없음 → 종료

총 144개 수집 완료


In [ ]:
df = pd.DataFrame(all_items)
df.to_csv('compose_coffee_menu.csv', index=False, encoding='utf-8-sig')

print(f"저장 완료 → compose_coffee_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → compose_coffee_menu.csv

카테고리별 수집 수:
category
시즌한정         33
커피 · 더치      31
티            31
논커피 라떼       18
프라페 · 스무디    12
에이드 · 주스     12
밀크쉐이크         7
Name: count, dtype: int64

샘플 데이터:
   brand                 name category  is_coffee price
0  컴포즈커피  ICE THE CITY 생초콜릿라떼     시즌한정      False  None
1  컴포즈커피   ICE THE CITY 올데이오트     시즌한정      False  None
2  컴포즈커피   HOT THE CITY 올데이오트     시즌한정      False  None
3  컴포즈커피        ICE 에어리 아메리카노     시즌한정      False  None
4  컴포즈커피              ICE 매샷추     시즌한정      False  None
5  컴포즈커피           말차샷 유자 스무디     시즌한정      False  None
6  컴포즈커피         ICE 크림 말차 라떼     시즌한정      False  None
7  컴포즈커피        ICE 어센틱 말차 라떼     시즌한정      False  None
8  컴포즈커피        HOT 어센틱 말차 라떼     시즌한정      False  None
9  컴포즈커피       ICE 에어레이팅 꿀 말차     시즌한정      False  None


## 3. 빽다방

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

# 커피로 볼 카테고리
COFFEE_CATEGORIES = {'커피'}

# 수집 제외 카테고리
EXCLUDE_CATEGORIES = {'아이스크림/디저트'}

CATEGORIES = {
    '커피': 'https://paikdabang.com/menu/menu_coffee/',
    '음료': 'https://paikdabang.com/menu/menu_drink/',
    '빽스치노': 'https://paikdabang.com/menu/menu_ccino/',
}

print("설정 완료")

✅ 설정 완료


In [ ]:
import re

def parse_paikdabang(soup, cat_name):
    items = []
    menu_list = soup.find('div', class_='menu_list')
    if not menu_list:
        return items

    for li in menu_list.find_all('li'):
        try:
            # 메뉴명
            name = li.find('p', class_='menu_tit')
            if not name:
                continue
            name = name.get_text(strip=True)

            # 영문명
            eng = li.find('div', class_='menu_tit2')
            eng_name = eng.get_text(strip=True) if eng else ''

            # ICE / HOT (메뉴명에서 추출)
            if '(ICED)' in name or 'ICED' in name.upper():
                temp = 'ICE'
            elif '(HOT)' in name or 'HOT' in name.upper():
                temp = 'HOT'
            else:
                temp = ''

            # 사이즈 (oz 추출)
            size_tag = li.find('p', class_='menu_ingredient_basis')
            size = ''
            if size_tag:
                # 여러 개일 수 있으니 전체 탐색
                for p in li.find_all('p', class_='menu_ingredient_basis'):
                    text = p.get_text(strip=True)
                    match = re.search(r'(\d+)\s*oz', text)
                    if match:
                        size = match.group(1) + 'oz'
                        break

            # 커피 여부 (고카페인 or 카테고리 기준)
            txt = li.find('p', class_='txt')
            txt_text = txt.get_text(strip=True) if txt else ''
            is_coffee = cat_name in COFFEE_CATEGORIES or '고카페인' in txt_text

            items.append({
                'brand': '빽다방',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'size': size,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [22]:
all_items = []

for cat_name, url in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')
    items = parse_paikdabang(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")
    time.sleep(1)

print(f"\n총 {len(all_items)}개 수집 완료")


▶ 커피 수집 시작
  108개 수집

▶ 음료 수집 시작
  143개 수집

▶ 빽스치노 수집 시작
  18개 수집

총 269개 수집 완료


In [ ]:
df = pd.DataFrame(all_items)
df.to_csv('paikdabang_menu.csv', index=False, encoding='utf-8-sig')

print("저장 완료 → paikdabang_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → paikdabang_menu.csv

카테고리별 수집 수:
category
음료      143
커피      108
빽스치노     18
Name: count, dtype: int64

샘플 데이터:
  brand              name                        name_en temp  size category  \
0   빽다방   에어폼 아메리카노(ICED)             AIR FOAM AMERICANO  ICE  24oz       커피   
1   빽다방           더블에스프레소                double espresso                  커피   
2   빽다방      디카페인 더블에스프레소          DECAF DOUBLE ESPRESSO                  커피   
3   빽다방        아메리카노(HOT)                      AMERICANO  HOT  16oz       커피   
4   빽다방       아메리카노(ICED)                      AMERICANO  ICE  24oz       커피   
5   빽다방   디카페인 아메리카노(HOT)                DECAF AMERICANO  HOT  16oz       커피   
6   빽다방  디카페인 아메리카노(ICED)                DECAF AMERICANO  ICE  24oz       커피   
7   빽다방     레드불 꿀샷추(ICED)  RED BULL with HONEY, ESPRESSO  ICE  24oz       커피   
8   빽다방         원조커피(HOT)          original mixed coffee  HOT  16oz       커피   
9   빽다방        원조커피(ICED)          ORIGINAL MIXED COFFEE  ICE  24oz       커피 

## 4. 이디야

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import re
import time

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

# 커피로 볼 카테고리
COFFEE_CATEGORIES = {'COFFEE', 'DECAF'}

# 수집 제외 카테고리
EXCLUDE_CATEGORIES = {'ICE CREAM', 'TOPPING', 'RTD'}

CATEGORIES = {
    'COFFEE': '12',
    'BEVERAGE': '13',
    'BLENDING TEA': '14',
    'FLATCCINO': '15',
    'SHAKE & ADE': '16',
    'ICE FLAKES': '71',
    'DECAF': '155',
}

print("설정 완료")

✅ 설정 완료


In [ ]:
def parse_ediya(soup, cat_name):
    items = []
    menu_ul = soup.find('ul', id='menu_ul')
    if not menu_ul:
        return items

    for li in menu_ul.find_all('li'):
        try:
            # 메뉴명 (menu_tt에서)
            name_tag = li.find('div', class_='menu_tt')
            if not name_tag:
                continue
            span = name_tag.find('span')
            name = span.get_text(strip=True) if span else name_tag.get_text(strip=True)

            # 영문명 (h2 안의 span)
            h2 = li.find('h2')
            eng_name = ''
            if h2:
                span = h2.find('span')
                eng_name = span.get_text(strip=True) if span else ''

            # temp (메뉴명에서 추출)
            if 'ICED' in name.upper():
                temp = 'ICE'
            elif 'HOT' in name.upper():
                temp = 'HOT'
            else:
                temp = ''

            # 사이즈 (메뉴명 앞 (L), (EX) + pro_size ml)
            size_label = ''
            if name.startswith('(L)'):
                size_label = 'L'
            elif name.startswith('(EX)'):
                size_label = 'EX'

            size_tag = li.find('div', class_='pro_size')
            size_ml = ''
            if size_tag:
                match = re.search(r'(\d+)ml', size_tag.get_text())
                size_ml = match.group(1) + 'ml' if match else ''

            size = f"{size_label} {size_ml}".strip()

            # 커피 여부
            is_coffee = cat_name in COFFEE_CATEGORIES

            items.append({
                'brand': '이디야',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'size': size,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [41]:
all_items = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, cat_value in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://www.ediya.com/contents/drink.html")
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.ID, "menu_ul"))
    )
    time.sleep(2)

    # 카테고리 체크박스 클릭
    checkbox = driver.find_element(By.CSS_SELECTOR, f'input[value="{cat_value}"]')
    driver.execute_script("arguments[0].click();", checkbox)
    time.sleep(2)

    # 더보기 버튼 계속 클릭
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'div.con_btn a')
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1)
            else:
                break
        except:
            break

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = parse_ediya(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ COFFEE 수집 시작
  66개 수집

▶ BEVERAGE 수집 시작
  66개 수집

▶ BLENDING TEA 수집 시작
  74개 수집

▶ FLATCCINO 수집 시작
  20개 수집

▶ SHAKE & ADE 수집 시작
  24개 수집

▶ ICE FLAKES 수집 시작
  0개 수집

▶ DECAF 수집 시작
  65개 수집

총 315개 수집 완료


In [43]:
df = pd.DataFrame(all_items)
df.to_csv('ediya_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → ediya_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → ediya_menu.csv

카테고리별 수집 수:
category
BLENDING TEA    74
COFFEE          66
BEVERAGE        66
DECAF           65
SHAKE & ADE     24
FLATCCINO       20
Name: count, dtype: int64

샘플 데이터:
  brand                  name                           name_en temp  \
0   이디야      (L) HOT 카페 아메리카노            (L) HOT Cafe Americano  HOT   
1   이디야     (EX) HOT 카페 아메리카노           (EX) HOT Cafe Americano  HOT   
2   이디야     (L) ICED 카페 아메리카노           (L) ICED Cafe Americano  ICE   
3   이디야    (EX) ICED 카페 아메리카노          (EX) ICED Cafe Americano  ICE   
4   이디야          (L) HOT 달달커피              (L) HOT Mixed coffee  HOT   
5   이디야         (EX) HOT 달달커피             (EX) HOT Mixed coffee  HOT   
6   이디야     (L) HOT 제로슈가 달달커피   (L) HOT Zero sugar Mixed Coffee  HOT   
7   이디야    (EX) HOT 제로슈가 달달커피  (EX) HOT Zero sugar Mixed Coffee  HOT   
8   이디야   (L) ICED 헤이즐넛 아메리카노       (L) ICED Hazelnut Americano  ICE   
9   이디야  (EX) ICED 헤이즐넛 아메리카노      (EX) ICED Hazelnut Americano  ICE   

       size

## 5. 투썸

In [44]:
url = "https://mo.twosome.co.kr/mn/menuInfoList.do"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 31149


In [50]:
COFFEE_SUBCATEGORIES = {'커피'}
EXCLUDE_SUBCATEGORIES = {'아이스크림/빙수', 'NEW'}

SUBCATEGORIES = {
    '커피': '01',
    '음료': '02',
    '티/티라떼': '03',
}

print("✅ 설정 완료")

✅ 설정 완료


In [51]:
def parse_twosome(soup, cat_name):
    items = []
    menu_list = soup.find('ul', class_='ui-goods-list-default')
    if not menu_list:
        return items

    for li in menu_list.find_all('li'):
        try:
            name_tag = li.find('p', class_='menu-title')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            items.append({
                'brand': '투썸플레이스',
                'name': name,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_SUBCATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [52]:
all_items = []
driver = webdriver.Chrome(service=service, options=options)

driver.get("https://mo.twosome.co.kr/mn/menuInfoList.do")
time.sleep(3)

# 커피/음료 대카테고리 클릭
main_tab = driver.find_element(By.CSS_SELECTOR, 'a.tab[grtval="1"]')
driver.execute_script("arguments[0].click();", main_tab)
time.sleep(2)

for cat_name, cat_value in SUBCATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")

    # 서브카테고리 클릭
    sub_tab = driver.find_element(By.CSS_SELECTOR, f'a.tab[midval="{cat_value}"]')
    driver.execute_script("arguments[0].click();", sub_tab)
    time.sleep(2)

    # 더보기 버튼 있으면 계속 클릭
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'a.btn-more')
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1)
            else:
                break
        except:
            break

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = parse_twosome(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ 커피 수집 시작
  28개 수집

▶ 음료 수집 시작
  29개 수집

▶ 티/티라떼 수집 시작
  19개 수집

총 76개 수집 완료


In [53]:
df = pd.DataFrame(all_items)
df.to_csv('twosome_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → twosome_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → twosome_menu.csv

카테고리별 수집 수:
category
음료       29
커피       28
티/티라떼    19
Name: count, dtype: int64

샘플 데이터:
    brand          name category  is_coffee price
0  투썸플레이스  생크림 말차 카페 라떼       커피       True  None
1  투썸플레이스     생크림 카페 라떼       커피       True  None
2  투썸플레이스     생크림 아메리카노       커피       True  None
3  투썸플레이스  피스타치오 초콜릿 모카       커피       True  None
4  투썸플레이스  디카페인 콜드브루 라떼       커피       True  None
5  투썸플레이스       바닐라빈 라떼       커피       True  None
6  투썸플레이스         아메리카노       커피       True  None
7  투썸플레이스         카페 라떼       커피       True  None
8  투썸플레이스          카푸치노       커피       True  None
9  투썸플레이스        바닐라 라떼       커피       True  None


## 6. 매머드 커피

In [54]:
url = "https://mmthcoffee.com/sub/menu/list_coffee_sub.php?menuType=C"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 21493


In [66]:
COFFEE_CATEGORIES = {'커피', '콜드브루', '32oz'}

CATEGORIES = {
    '32oz': 'O',
    '커피': 'C',
    '콜드브루': 'D',
    '논커피': 'N',
    '티·에이드': 'T',
    '프라페·블렌디드': 'B',
}

BASE_URL = "https://mmthcoffee.com/sub/menu/list_coffee_sub.php?menuType={}"
print("✅ 설정 완료")

✅ 설정 완료


In [72]:
import re

def parse_size_from_popup(driver):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    table = soup.find('div', class_='i_table')
    temp, size = '', ''
    
    if table:
        # 헤더에서 HOT/ICE 직접 찾기
        headers = table.find_all('th')
        hot_size, ice_size = '', ''
        
        for th in headers:
            text = th.get_text(strip=True)
            hot_match = re.search(r'HOT\((\d+oz)\)', text)
            ice_match = re.search(r'ICE\((\d+oz)\)', text)
            if hot_match:
                hot_size = hot_match.group(1)
            if ice_match:
                ice_size = ice_match.group(1)

        if hot_size and ice_size:
            temp = 'HOT/ICE'
            size = f"HOT({hot_size})/ICE({ice_size})"
        elif hot_size:
            temp = 'HOT'
            size = f"HOT({hot_size})"
        elif ice_size:
            temp = 'ICE'
            size = f"ICE({ice_size})"

    return temp, size

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [76]:
all_items = []
driver = webdriver.Chrome(service=service)

for cat_name, menu_type in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    url = BASE_URL.format(menu_type)
    driver.get(url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    menu_items = soup.find_all('li', class_=lambda x: x and 'animation' in x)
    print(f"  메뉴 수: {len(menu_items)}개")

    for i, li in enumerate(menu_items):
        try:
            name_tag = li.find('strong')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            eng_tag = li.find('p', class_='eng')
            eng_name = eng_tag.get_text(strip=True) if eng_tag else ''

            # 메뉴 클릭해서 팝업 열기
            a_tag = driver.find_elements(By.CSS_SELECTOR, 'li.animation a')[i]
            driver.execute_script("arguments[0].click();", a_tag)
            time.sleep(1)

            # 사이즈 수집
            temp, size = parse_size_from_popup(driver)

            # 닫기 버튼 클릭
            try:
                close_btn = driver.find_element(By.CSS_SELECTOR, 'button.close_b')
                driver.execute_script("arguments[0].click();", close_btn)
                time.sleep(1)
            except Exception as e:
                print(f"  닫기 버튼 에러: {e}")

            all_items.append({
                'brand': '매머드커피',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'size': size,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_CATEGORIES,
                'price': None
            })
            print(f"  [{i+1}] {name} | HOT:{temp} SIZE:{size}")

        except Exception as e:
            print(f"  [{i+1}] 에러: {e}")
            continue

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ 32oz 수집 시작
  메뉴 수: 9개
  [1] 매머드 커피 | HOT:ICE SIZE:ICE(32oz)
  [2] 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [3] 아샷추 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [4] 스노우 매머드 커피 | HOT:ICE SIZE:ICE(32oz)
  [5] 허니베리 홍초 에이드 | HOT:ICE SIZE:ICE(32oz)
  [6] 패션 오렌지 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [7] 디카페인 매머드 커피 | HOT:ICE SIZE:ICE(32oz)
  [8] 디카페인 아샷추 아이스티 | HOT:ICE SIZE:ICE(32oz)
  [9] 디카페인 스노우 매머드 커피 | HOT:ICE SIZE:ICE(32oz)

▶ 커피 수집 시작
  메뉴 수: 37개
  [1] 아메리카노 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [2] 꿀 커피 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [3] 아몬드 아메리카노 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [4] 아샷추 아이스티 | HOT:ICE SIZE:ICE(20oz)
  [5] 카페 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [6] 카푸치노 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [7] 꿀 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [8] 아몬드 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [9] 바닐라 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [10] 꿀바나 라떼 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [11] 카라멜 마키아토 | HOT:HOT/ICE SIZE:HOT(16oz)/ICE(20oz)
  [12] 카페 모카 | 

In [78]:
df = pd.DataFrame(all_items)
df.to_csv('mammoth_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → mammoth_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → mammoth_menu.csv

카테고리별 수집 수:
category
커피          37
티·에이드       32
프라페·블렌디드    15
콜드브루        14
32oz         9
논커피          8
Name: count, dtype: int64

샘플 데이터:
   brand             name                            name_en     temp  \
0  매머드커피           매머드 커피             Mammoth Iced Americano      ICE   
1  매머드커피             아이스티                     Peach Iced Tea      ICE   
2  매머드커피         아샷추 아이스티           Peach Iced Tea(add shot)      ICE   
3  매머드커피       스노우 매머드 커피        Snow Mammoth Iced Americano      ICE   
4  매머드커피      허니베리 홍초 에이드        Honey Berry Red Vinegar Ade      ICE   
5  매머드커피      패션 오렌지 아이스티            Passion Orange Iced Tea      ICE   
6  매머드커피      디카페인 매머드 커피       Decaf Mammoth Iced Americano      ICE   
7  매머드커피    디카페인 아샷추 아이스티     Decaf Peach Iced Tea(add shot)      ICE   
8  매머드커피  디카페인 스노우 매머드 커피  Decaf Snow Mammoth Iced Americano      ICE   
9  매머드커피            아메리카노                          Americano  HOT/ICE   

                  size

## 7. 공차

In [4]:
url = "https://www.theventi.co.kr/new2022/menu/all.html"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 54626


In [10]:
COFFEE_CATEGORIES = {'커피', '디카페인'}
EXCLUDE_CATEGORIES = {'사이드메뉴/RTD', '신메뉴'}

CATEGORIES = {
    '커피': '2',
    '디카페인': '3',
    '아이스 블렌디드': '4',
    '주스/에이드': '5',
    '버블티/티': '6',
    '베버리지': '7',
}

BASE_URL = "https://www.theventi.co.kr/new2022/menu/all.html?mode={}"
print("✅ 설정 완료")

✅ 설정 완료


In [18]:
import re

def parse_venti(soup, cat_name):
    items = []
    menu_list = soup.find('div', class_='menu_list')
    if not menu_list:
        return items

    for li in menu_list.find_all('li', class_='item'):
        try:
            name_tag = li.find('p', class_='tit')
            if not name_tag:
                continue
            full_name = name_tag.get_text(strip=True)

            # 사이즈 추출 - 괄호 안 내용
            size_match = re.search(r'\(([^)]+)\)', full_name)
            size = size_match.group(1) if size_match else ''

            # 사이즈 괄호 제거한 메뉴명
            name = re.sub(r'\s*\([^)]+\)', '', full_name).strip()

            # ICE/HOT 구분
            type_tag = li.find('p', class_='type')
            temp = ''
            if type_tag:
                has_hot = bool(type_tag.find('i', class_='hot'))
                has_ice = bool(type_tag.find('i', class_='ice'))
                if has_hot and has_ice:
                    temp = 'HOT/ICE'
                elif has_hot:
                    temp = 'HOT'
                elif has_ice:
                    temp = 'ICE'

            items.append({
                'brand': '더벤티',
                'name': name,
                'size': size,
                'temp': temp,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_CATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [19]:
import time

all_items = []

for cat_name, mode in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    url = BASE_URL.format(mode)
    res = requests.get(url, headers=headers)
    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, 'html.parser')

    items = parse_venti(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")
    time.sleep(1)

print(f"\n총 {len(all_items)}개 수집 완료")


▶ 커피 수집 시작
  18개 수집

▶ 디카페인 수집 시작
  18개 수집

▶ 아이스 블렌디드 수집 시작
  18개 수집

▶ 주스/에이드 수집 시작
  8개 수집

▶ 버블티/티 수집 시작
  12개 수집

▶ 베버리지 수집 시작
  13개 수집

총 87개 수집 완료


In [20]:
df = pd.DataFrame(all_items)
df.to_csv('theventi_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → venti_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → venti_menu.csv

카테고리별 수집 수:
category
커피          18
디카페인        18
아이스 블렌디드    18
베버리지        13
버블티/티       12
주스/에이드       8
Name: count, dtype: int64

샘플 데이터:
  brand       name    size     temp category  is_coffee price
0   더벤티    핫 아메리카노      라지      HOT       커피       True  None
1   더벤티  아이스 아메리카노   라지/점보      ICE       커피       True  None
2   더벤티       카페라떼   라지/점보  HOT/ICE       커피       True  None
3   더벤티     오트카페라떼   라지/점보  HOT/ICE       커피       True  None
4   더벤티     바닐라딥라떼   라지/점보  HOT/ICE       커피       True  None
5   더벤티       믹스커피   라지/점보  HOT/ICE       커피       True  None
6   더벤티    헤이즐넛딥라떼   라지/점보  HOT/ICE       커피       True  None
7   더벤티       연유라떼   라지/점보  HOT/ICE       커피       True  None
8   더벤티      아인슈페너  미디엄/라지      ICE       커피       True  None
9   더벤티      코코넛라떼   라지/점보  HOT/ICE       커피       True  None


## 8. 공차

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import requests
import pandas as pd
import time
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
}

COFFEE_CATEGORIES = {'커피'}
CATEGORIES = {
    '베스트셀러': '001002',
    '밀크티': '001006',
    '스무디': '001010',
    '오리지널 티': '001003',
    '프룻티&모어': '001015',
    '커피': '001011',
}

BASE_URL = "https://www.gong-cha.co.kr"
print("✅ 설정 완료")

✅ 설정 완료


In [35]:
def parse_detail(detail_url):
    res = requests.get(detail_url, headers=headers, verify=False)
    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, 'html.parser')

    results = []
    table = soup.find('div', class_='table-list')
    if not table:
        return [{'temp': '', 'size': ''}]

    tbody = table.find('tbody')
    if not tbody:
        return [{'temp': '', 'size': ''}]

    for row in tbody.find_all('tr'):
        cells = row.find_all('td')
        if not cells:
            continue

        # temp (Cold/Hot)
        temp_cell = row.find('td', class_='border-none row')
        if temp_cell:
            temp_text = temp_cell.get_text(strip=True)
            temp = 'ICE' if 'Cold' in temp_text else 'HOT' if 'Hot' in temp_text else ''
        else:
            # rowspan으로 합쳐진 경우 이전 temp 유지
            temp = results[-1]['temp'] if results else ''

        # size
        size = cells[1].get_text(strip=True) if len(cells) > 1 else ''

        results.append({'temp': temp, 'size': size})

    return results if results else [{'temp': '', 'size': ''}]

print("✅ 상세 파싱 함수 준비 완료")

✅ 상세 파싱 함수 준비 완료


In [36]:
all_items = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, cat_code in CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get(f"{BASE_URL}/brand/menu/product?category=001001&scroll=y")
    time.sleep(3)

    # 카테고리 탭 클릭
    tab = driver.find_element(By.CSS_SELECTOR, f'a[data-cate="{cat_code}"]')
    driver.execute_script("arguments[0].click();", tab)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    menu_items = soup.find_all('li', class_='scroll-item observed')
    print(f"  메뉴 수: {len(menu_items)}개")

    for i, li in enumerate(menu_items):
        try:
            name_tag = li.find('p')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            # 상세 페이지 URL
            a_tag = li.find('a')
            detail_href = a_tag.get('href', '') if a_tag else ''
            detail_url = BASE_URL + detail_href

            # 상세 페이지에서 temp/size 수집
            details = parse_detail(detail_url)
            time.sleep(0.5)

            for detail in details:
                all_items.append({
                    'brand': '공차',
                    'name': name,
                    'temp': detail['temp'],
                    'size': detail['size'],
                    'category': cat_name,
                    'is_coffee': cat_name in COFFEE_CATEGORIES,
                    'price': None
                })

            print(f"  [{i+1}] {name} | {details}")

        except Exception as e:
            print(f"  [{i+1}] 에러: {e}")
            continue

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ 베스트셀러 수집 시작
  메뉴 수: 11개
  [1] 블랙 밀크티 + 펄 1L | [{'temp': 'ICE', 'size': 'J'}]
  [2] 허니 자몽 블랙티 | [{'temp': 'ICE', 'size': 'L'}, {'temp': 'ICE', 'size': '651'}, {'temp': 'HOT', 'size': 'L'}, {'temp': 'HOT', 'size': '473'}]
  [3] 미니펄 망고 밀크 + 리얼 망고 | [{'temp': 'ICE', 'size': 'L'}, {'temp': 'ICE', 'size': '651'}]
  [4] 초콜렛 밀크티 + 치즈폼 | [{'temp': 'ICE', 'size': 'L'}, {'temp': 'ICE', 'size': '651'}, {'temp': 'HOT', 'size': 'L'}, {'temp': 'HOT', 'size': '473'}]
  [5] 제주 그린 밀크티 + 펄 | [{'temp': 'ICE', 'size': 'L'}, {'temp': 'ICE', 'size': '651'}, {'temp': 'HOT', 'size': 'L'}, {'temp': 'HOT', 'size': '473'}]
  [6] 아메리카노 | [{'temp': 'ICE', 'size': 'L'}, {'temp': 'ICE', 'size': '651'}, {'temp': 'HOT', 'size': 'L'}, {'temp': 'HOT', 'size': '473'}]
  [7] 딸기 쿠키 스무디 + 펄 | [{'temp': 'ICE', 'size': 'L'}]
  [8] 우롱티 + 코코넛 + 밀크폼 | [{'temp': 'ICE', 'size': 'L'}, {'temp': 'ICE', 'size': '651'}, {'temp': 'HOT', 'size': 'L'}, {'temp': 'HOT', 'size': '473'}]
  [9] 망고 요구르트 + 화이트펄 | [{'temp': 'ICE', 'size': 'L'},

In [37]:
df = pd.DataFrame(all_items)
df.to_csv('gongcha_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → gongcha_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → gongcha_menu.csv

카테고리별 수집 수:
category
밀크티       41
베스트셀러     34
커피        23
스무디       16
오리지널 티    16
Name: count, dtype: int64

샘플 데이터:
  brand               name temp size category  is_coffee price
0    공차      블랙 밀크티 + 펄 1L  ICE    J    베스트셀러      False  None
1    공차          허니 자몽 블랙티  ICE    L    베스트셀러      False  None
2    공차          허니 자몽 블랙티  ICE  651    베스트셀러      False  None
3    공차          허니 자몽 블랙티  HOT    L    베스트셀러      False  None
4    공차          허니 자몽 블랙티  HOT  473    베스트셀러      False  None
5    공차  미니펄 망고 밀크 + 리얼 망고  ICE    L    베스트셀러      False  None
6    공차  미니펄 망고 밀크 + 리얼 망고  ICE  651    베스트셀러      False  None
7    공차      초콜렛 밀크티 + 치즈폼  ICE    L    베스트셀러      False  None
8    공차      초콜렛 밀크티 + 치즈폼  ICE  651    베스트셀러      False  None
9    공차      초콜렛 밀크티 + 치즈폼  HOT    L    베스트셀러      False  None


## 9. 커피베이

In [3]:
url = "https://www.coffeebay.com/menu/coffeebay"
res = requests.get(url, headers=headers)
print("상태 코드:", res.status_code)
print("페이지 길이:", len(res.text))

상태 코드: 200
페이지 길이: 38175


In [17]:
COFFEE_CATEGORIES = {'커피', '베이직라떼'}
EXCLUDE_CATEGORIES = {'NEW', '전체', '디저트', '베이커리', 'MD'}

CATEGORIES = ['커피', '베이직라떼', '프라노베&스무벨라', '에이드', '주스&티']

print("✅ 설정 완료")

✅ 설정 완료


In [18]:
def parse_coffeebay(soup, cat_name):
    items = []

    # h3 태그에서 메뉴명 수집
    for h3 in soup.find_all('h3', class_='text-base font-medium text-gray-900'):
        try:
            name = h3.get_text(strip=True)
            if not name:
                continue

            # 메뉴명에서 temp 추출
            if name.startswith('아이스') or 'ICE' in name.upper():
                temp = 'ICE'
            else:
                temp = 'HOT'

            items.append({
                'brand': '커피베이',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': cat_name in COFFEE_CATEGORIES,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

print("✅ 파싱 함수 준비 완료")

✅ 파싱 함수 준비 완료


In [19]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time

service = Service(r"C:\Users\pyk71\KIO_crawling\chromedriver.exe")
options = webdriver.ChromeOptions()
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

all_items = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name in CATEGORIES:
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://www.coffeebay.com/menu/coffeebay")
    time.sleep(3)

    # 카테고리 버튼 찾아서 클릭
    buttons = driver.find_elements(By.CSS_SELECTOR, 'button span.text-base.font-medium')
    for btn in buttons:
        if btn.text.strip() == cat_name:
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)
            break

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = parse_coffeebay(soup, cat_name)
    all_items.extend(items)
    print(f"  {len(items)}개 수집")

driver.quit()
print(f"\n총 {len(all_items)}개 수집 완료")


▶ 커피 수집 시작
  25개 수집

▶ 베이직라떼 수집 시작
  23개 수집

▶ 프라노베&스무벨라 수집 시작
  16개 수집

▶ 에이드 수집 시작
  5개 수집

▶ 주스&티 수집 시작
  33개 수집

총 102개 수집 완료


In [20]:
df = pd.DataFrame(all_items)
df.to_csv('coffeebay_menu.csv', index=False, encoding='utf-8-sig')

print("✅ 저장 완료 → coffeebay_menu.csv")
print(f"\n카테고리별 수집 수:")
print(df['category'].value_counts())
print(f"\n샘플 데이터:")
print(df.head(10))

✅ 저장 완료 → coffeebay_menu.csv

카테고리별 수집 수:
category
주스&티         33
커피           25
베이직라떼        23
프라노베&스무벨라    16
에이드           5
Name: count, dtype: int64

샘플 데이터:
  brand        name temp category  is_coffee price
0  커피베이        아포가토  HOT       커피       True  None
1  커피베이       아인슈페너  HOT       커피       True  None
2  커피베이        콘슈페너  HOT       커피       True  None
3  커피베이       옛날 커피  HOT       커피       True  None
4  커피베이       돌체 라떼  HOT       커피       True  None
5  커피베이   아이스 돌체 라떼  ICE       커피       True  None
6  커피베이       에스프레소  HOT       커피       True  None
7  커피베이   아이스 아메리카노  ICE       커피       True  None
8  커피베이       아메리카노  HOT       커피       True  None
9  커피베이  콜드브루 아메리카노  HOT       커피       True  None


## 10. 카페베네

In [24]:
# 카페베네 설정
CAFFEBENE_CATEGORIES = {
    '커피': 'http://www.caffebene.co.kr/menu/menu_list.html?code=001000',
    '음료': 'http://www.caffebene.co.kr/menu/menu_list.html?code=002000',
}
CAFFEBENE_COFFEE = {'커피'}

# 카페베네 수집
all_items_caffebene = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, url in CAFFEBENE_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    menu_items = soup.select('ul.menu-list-items li')
    print(f"  메뉴 수: {len(menu_items)}개")

    for li in menu_items:
        try:
            name_tag = li.find('div', class_='m-name')
            if not name_tag:
                continue
            full_name = name_tag.get_text(strip=True)

            # temp 추출
            if '(HOT)' in full_name:
                temp = 'HOT'
            elif '(ICED)' in full_name or '(ICE)' in full_name:
                temp = 'ICE'
            else:
                temp = ''

            # temp 괄호 제거한 메뉴명
            name = full_name.replace('(HOT)', '').replace('(ICED)', '').replace('(ICE)', '').strip()

            all_items_caffebene.append({
                'brand': '카페베네',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': cat_name in CAFFEBENE_COFFEE,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    print(f"  {len(all_items_caffebene)}개 누적")

driver.quit()

# CSV 저장
df = pd.DataFrame(all_items_caffebene)
df.to_csv('caffebene_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_caffebene)}개 → caffebene_menu.csv")
print(df['category'].value_counts())


▶ 커피 수집 시작
  메뉴 수: 25개
  25개 누적

▶ 음료 수집 시작
  메뉴 수: 49개
  74개 누적

✅ 완료! 총 74개 → caffebene_menu.csv
category
음료    49
커피    25
Name: count, dtype: int64


## 11. 할리스

In [37]:
def parse_hollys(soup, cat_name):
    items = []

    for table in soup.find_all('table', summary=True):
        try:
            name = table.get('summary', '').strip()
            if not name or name == '':
                continue

            # temp 추출
            text = table.get_text(strip=True)
            has_hot = 'HOT' in text
            has_ice = 'ICED' in text
            if has_hot and has_ice:
                temp = 'HOT/ICE'
            elif has_hot:
                temp = 'HOT'
            elif has_ice:
                temp = 'ICE'
            else:
                temp = ''

            items.append({
                'brand': '할리스',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': cat_name in HOLLYS_COFFEE,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue
    return items

# 수집 실행
all_items_hollys = []
for cat_name, url in HOLLYS_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    res = requests.get(url, headers=headers)
    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, 'html.parser')
    items = parse_hollys(soup, cat_name)
    all_items_hollys.extend(items)
    print(f"  {len(items)}개 수집")
    time.sleep(1)

df = pd.DataFrame(all_items_hollys)
df.to_csv('hollys_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_hollys)}개 → hollys_menu.csv")
print(df['category'].value_counts())
print(df.head(10))


▶ COFFEE 수집 시작
  26개 수집

▶ 라떼·초콜릿·티 수집 시작
  24개 수집

▶ 할리치노·빙수 수집 시작
  7개 수집

▶ 스무디·주스 수집 시작
  8개 수집

▶ 스파클링 수집 시작
  5개 수집

✅ 완료! 총 70개 → hollys_menu.csv
category
COFFEE      26
라떼·초콜릿·티    24
스무디·주스       8
할리치노·빙수      7
스파클링         5
Name: count, dtype: int64
  brand                   name     temp category  is_coffee price
0   할리스              두쫀크 아인슈페너      ICE   COFFEE       True  None
1   할리스               두쫀크 돌체라떼      ICE   COFFEE       True  None
2   할리스            저당 바닐라 딜라이트  HOT/ICE   COFFEE       True  None
3   할리스         바닐라 딜라이트 아인슈페너      ICE   COFFEE       True  None
4   할리스  디카페인 콜드브루 저당 바닐라 딜라이트  HOT/ICE   COFFEE       True  None
5   할리스                  유자리카노      ICE   COFFEE       True  None
6   할리스          디카페인 콜드브루 아샷추      ICE   COFFEE       True  None
7   할리스                    아샷추      ICE   COFFEE       True  None
8   할리스              몬스터 아메리카노      ICE   COFFEE       True  None
9   할리스         블랙아리아<br>아메리카노  HOT/ICE   COFFEE       True  None


## 12. 탐앤탐스

In [ ]:
# temp가 없음 그림으로만 표시되어있음;;;;

all_items_tomntoms = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name in TOMNTOMS_CATEGORIES:
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://www.tomntoms.com/menu/drink")
    time.sleep(3)

    # 전체 체크박스 해제
    try:
        all_checkbox = driver.find_element(By.CSS_SELECTOR, 'ul.search-group li:first-child input')
        driver.execute_script("arguments[0].click();", all_checkbox)
        time.sleep(1)
    except Exception as e:
        print(f"  전체 해제 에러: {e}")

    # 해당 카테고리 input 클릭
    try:
        lis = driver.find_elements(By.CSS_SELECTOR, 'ul.search-group li')
        for li in lis:
            text = li.text.strip()
            if text == cat_name:
                inp = li.find_element(By.CSS_SELECTOR, 'input')
                driver.execute_script("arguments[0].click();", inp)
                time.sleep(2)
                break
    except Exception as e:
        print(f"  카테고리 클릭 에러: {e}")
        continue

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    grid = soup.find('div', class_=lambda x: x and 'grid' in x and 'grid-cols' in x)
    if not grid:
        print("  메뉴 없음")
        continue

    for item in grid.find_all('div', class_='relative w-full'):
        try:
            name_tag = item.find('span', class_='tracking-wider')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)
            eng_tag = item.find('h3', class_='text-sm text-gray-400')
            eng_name = eng_tag.get_text(strip=True) if eng_tag else ''

            all_items_tomntoms.append({
                'brand': '탐앤탐스',
                'name': name,
                'name_en': eng_name,
                'category': cat_name,
                'is_coffee': cat_name in TOMNTOMS_COFFEE,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    print(f"  {len([x for x in all_items_tomntoms if x['category'] == cat_name])}개 수집")

driver.quit()

df = pd.DataFrame(all_items_tomntoms)
df.to_csv('tomntoms_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_tomntoms)}개 → tomntoms_menu.csv")
print(df['category'].value_counts())


▶ ESPRESSO 수집 시작
  3개 수집

▶ SINGLE ORIGIN 수집 시작
  4개 수집

▶ COLDBREW 수집 시작
  7개 수집

▶ BREW & COFFEE 수집 시작
  0개 수집

▶ AMERICANO & LATTE 수집 시작
  12개 수집

▶ LATTE & CHOCOLATE 수집 시작
  12개 수집

▶ TEA & ADE & JUICE 수집 시작
  12개 수집

▶ TOMNCHINO & COFFEE 수집 시작
  2개 수집

▶ TOMNCHINO & NON COFFEE 수집 시작
  8개 수집

▶ SMOOTHIE 수집 시작
  9개 수집

▶ SLUSH 수집 시작
  1개 수집

✅ 완료! 총 70개 → tomntoms_menu.csv
category
AMERICANO & LATTE         12
LATTE & CHOCOLATE         12
TEA & ADE & JUICE         12
SMOOTHIE                   9
TOMNCHINO & NON COFFEE     8
COLDBREW                   7
SINGLE ORIGIN              4
ESPRESSO                   3
TOMNCHINO & COFFEE         2
SLUSH                      1
Name: count, dtype: int64


## 13. 텐퍼센트

In [53]:
TENPERCENT_CATEGORIES = {
    '신메뉴': 'https://tenpercentcoffee.com/bbs/board.php?bo_table=new',
    '커피': 'https://tenpercentcoffee.com/bbs/board.php?bo_table=sub22',
    '음료/티': 'https://tenpercentcoffee.com/bbs/board.php?bo_table=sub23',
}
TENPERCENT_COFFEE = {'커피'}

all_items_tenpercent = []

for cat_name, base_url in TENPERCENT_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    page = 1
    while True:
        url = f"{base_url}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        gallery = soup.find('div', class_='gallery')
        if not gallery:
            break

        items = gallery.find_all('div', class_='bx-grandi')
        if not items:
            break

        for item in items:
            try:
                span = item.find('span')
                if not span:
                    continue
                name = span.get_text(strip=True)
                if not name:
                    continue

                all_items_tenpercent.append({
                    'brand': '텐퍼센트커피',
                    'name': name,
                    'temp': '',
                    'category': cat_name,
                    'is_coffee': cat_name in TENPERCENT_COFFEE,
                    'price': None
                })
            except Exception as e:
                print(f"  파싱 에러: {e}")
                continue

        # 다음 페이지 확인
        next_page_links = soup.select('a[href*="page="]')
        max_page = max([int(a['href'].split('page=')[-1]) for a in next_page_links if a['href'].split('page=')[-1].isdigit()], default=1)
        if page >= max_page:
            break
        page += 1
        time.sleep(1)

    print(f"  {len([x for x in all_items_tenpercent if x['category'] == cat_name])}개 수집")

df = pd.DataFrame(all_items_tenpercent)
df.to_csv('tenpercent_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_tenpercent)}개 → tenpercent_menu.csv")
print(df['category'].value_counts())


▶ 신메뉴 수집 시작
  48개 수집

▶ 커피 수집 시작
  18개 수집

▶ 음료/티 수집 시작
  45개 수집

✅ 완료! 총 111개 → tenpercent_menu.csv
category
신메뉴     48
음료/티    45
커피      18
Name: count, dtype: int64


## 14. 요거프레소

In [66]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

YOGER_CATEGORIES = {
    '커피&논커피': ('1', True),
    '요거트 라떼': ('34', False),
    '요거트 스무디': ('2', False),
    '요거트 쉐이크': ('36', False),
    '한입요거트': ('23', False),
    '토핑요거트': ('32', False),
    '에이드&티': ('35', False),
    '메리시리즈 & 빙수': ('3', False),
    'THE OTHER': ('33', False),
}

all_items = []

for cat_name, (cateno, is_coffee) in YOGER_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    url = f"https://www.yogerpresso.co.kr/menu/menu.html?cateno={cateno}"
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    items = soup.find_all('li', class_='item wow zoomIn')
    for item in items:
        try:
            name_tag = item.find('p', class_='text')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)
            all_items.append({
                'brand': '요거프레소',
                'name': name,
                'temp': None,
                'category': cat_name,
                'is_coffee': is_coffee,
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")
    time.sleep(0.5)

df = pd.DataFrame(all_items)
df.to_csv('yogerpresso_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → yogerpresso_menu.csv")
print(df['category'].value_counts())


▶ 커피&논커피 수집 시작
  34개 수집

▶ 요거트 라떼 수집 시작
  10개 수집

▶ 요거트 스무디 수집 시작
  9개 수집

▶ 요거트 쉐이크 수집 시작
  10개 수집

▶ 한입요거트 수집 시작
  31개 수집

▶ 토핑요거트 수집 시작
  47개 수집

▶ 에이드&티 수집 시작
  30개 수집

▶ 메리시리즈 & 빙수 수집 시작
  27개 수집

▶ THE OTHER 수집 시작
  62개 수집

✅ 완료! 총 260개 → yogerpresso_menu.csv
category
THE OTHER     62
토핑요거트         47
커피&논커피        34
한입요거트         31
에이드&티         30
메리시리즈 & 빙수    27
요거트 라떼        10
요거트 쉐이크       10
요거트 스무디        9
Name: count, dtype: int64


## 15. 하삼동커피

In [77]:
HASAMDONG_CATEGORIES = {
    '시그니처': ('120', True),
    '신메뉴/시즌메뉴': ('10', True),
    '커피': ('20', True),
    '콜드브루/디카페인': ('40', True),
    '보틀': ('30', True),
    '라떼': ('50', False),
    '스무디': ('60', False),
    '에이드': ('70', False),
    '주스/버블티': ('80', False),
    '차': ('110', False),
}

all_items = []
driver = webdriver.Chrome(service=service, options=options)
driver.get("https://www.hasamdongcoffee.com/menu_list.php")
time.sleep(3)

for cat_name, (cd, is_coffee) in HASAMDONG_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    try:
        btn = driver.find_element(By.CSS_SELECTOR, f'button.btnPdCtg[cd="{cd}"]')
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(2)
    except Exception as e:
        print(f"  카테고리 클릭 에러: {e}")
        continue

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    items = soup.find_all('li', class_=lambda x: x and f'menu{cd}' in x)

    for item in items:
        try:
            detail = item.find('div', class_='detail')
            if not detail:
                continue
            ps = detail.find_all('p')
            name = ps[0].get_text(strip=True) if len(ps) > 0 else ''
            temp = ps[1].get_text(strip=True) if len(ps) > 1 else None

            all_items.append({
                'brand': '하삼동커피',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('hasamdong_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → hasamdong_menu.csv")
print(df['category'].value_counts())


▶ 시그니처 수집 시작
  4개 수집

▶ 신메뉴/시즌메뉴 수집 시작
  10개 수집

▶ 커피 수집 시작
  20개 수집

▶ 콜드브루/디카페인 수집 시작
  9개 수집

▶ 보틀 수집 시작
  22개 수집

▶ 라떼 수집 시작
  12개 수집

▶ 스무디 수집 시작
  19개 수집

▶ 에이드 수집 시작
  6개 수집

▶ 주스/버블티 수집 시작
  14개 수집

▶ 차 수집 시작
  17개 수집

✅ 완료! 총 133개 → hasamdong_menu.csv
category
보틀           22
커피           20
스무디          19
차            17
주스/버블티       14
라떼           12
신메뉴/시즌메뉴     10
콜드브루/디카페인     9
에이드           6
시그니처          4
Name: count, dtype: int64


## 16. 팔공티

In [102]:
import re

PALGONG_MENUS = {
    '신메뉴': {
        '젤라또 라운지': ('신메뉴', 'gelato', False),
        '딸기': ('신메뉴', 'C12', False),
        '말차': ('신메뉴', 'matcha', False),
        '망고': ('신메뉴', 'Summer3', False),
        '제로/크림/아인슈페너': ('신메뉴', 'NEW PEACH', False),
        '우리나라 전통차': ('신메뉴', 'KOREA1', False),
        '80 MIX': ('신메뉴', 'MIX2', False),
    },
    '음료': {
        'GELATO LOUNGE': ('음료', 'gelato', False),
        'FRESH BREWED FRUIT TEA': ('음료', 'Coffee2', False),
        'MILK TEA': ('음료', 'Coffee11', False),
        'COFFEE': ('음료', 'Coffee5', True),
        'BREWED TEA': ('음료', 'Coffee3', False),
        'LATTE': ('음료', 'Coffee6', True),
        'ICE BLENDED': ('음료', 'Coffee9', False),
        'ICED TEA/ADE/JUICE': ('음료', 'coffee4', False),
        'TOPPING': ('음료', 'Coffee10', False),
    }
}

BASE_URL = "http://www.palgongtea.co.kr/goods/newest.html"

all_items = []

for menu_group, categories in PALGONG_MENUS.items():
    for cat_name, (menu, cate, is_coffee) in categories.items():
        print(f"\n▶ [{menu_group}] {cat_name} 수집 시작")
        page = 1

        while True:
            url = f"{BASE_URL}?menu={menu}&cate={cate}&page={page}"
            res = requests.get(url, headers=headers)
            soup = BeautifulSoup(res.text, 'html.parser')

            items = soup.find_all('li', class_='items')
            if not items:
                break

            for item in items:
                try:
                    name_tag = item.find('span', class_='itemNm')
                    if not name_tag:
                        continue
                    full_name = name_tag.get_text(strip=True)

                    # temp 파싱
                    temp_match = re.search(r'\((ICE|HOT)\)', full_name, re.IGNORECASE)
                    temp = temp_match.group(1).upper() if temp_match else None
                    name = re.sub(r'\s*\((ICE|HOT)\)', '', full_name, flags=re.IGNORECASE).strip()

                    all_items.append({
                        'brand': '팔공티',
                        'name': name,
                        'temp': temp,
                        'category': cat_name,
                        'menu_group': menu_group,
                        'is_coffee': is_coffee,
                    })
                except Exception as e:
                    print(f"  파싱 에러: {e}")
                    continue

            # 다음 페이지 확인
            next_page = soup.find('a', string=str(page + 1))
            if not next_page:
                break
            page += 1
            time.sleep(0.3)

        print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")
        time.sleep(0.5)

df = pd.DataFrame(all_items)
df.to_csv('palgong_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → palgong_menu.csv")
print(df.groupby(['menu_group', 'category']).size())


▶ [신메뉴] 젤라또 라운지 수집 시작
  13개 수집

▶ [신메뉴] 딸기 수집 시작
  5개 수집

▶ [신메뉴] 말차 수집 시작
  6개 수집

▶ [신메뉴] 망고 수집 시작
  2개 수집

▶ [신메뉴] 제로/크림/아인슈페너 수집 시작
  7개 수집

▶ [신메뉴] 우리나라 전통차 수집 시작
  8개 수집

▶ [신메뉴] 80 MIX 수집 시작
  3개 수집

▶ [음료] GELATO LOUNGE 수집 시작
  13개 수집

▶ [음료] FRESH BREWED FRUIT TEA 수집 시작
  6개 수집

▶ [음료] MILK TEA 수집 시작
  32개 수집

▶ [음료] COFFEE 수집 시작
  32개 수집

▶ [음료] BREWED TEA 수집 시작
  32개 수집

▶ [음료] LATTE 수집 시작
  6개 수집

▶ [음료] ICE BLENDED 수집 시작
  16개 수집

▶ [음료] ICED TEA/ADE/JUICE 수집 시작
  12개 수집

▶ [음료] TOPPING 수집 시작
  15개 수집

✅ 완료! 총 208개 → palgong_menu.csv
menu_group  category              
신메뉴         80 MIX                     3
            딸기                         5
            말차                         6
            망고                         2
            우리나라 전통차                   8
            제로/크림/아인슈페너                7
            젤라또 라운지                   13
음료          BREWED TEA                32
            COFFEE                    32
            FRESH BREWED FRUIT TEA     6
 

## 17. 아마스빈

In [107]:
soup = BeautifulSoup(res.text, 'html.parser')

AMASVIN_CATEGORIES = {
    '커피': True,
    '밀크티/라떼': False,
    '요거트/하동녹차/흑당': False,
    '쉐이크/스무디': False,
    '에이드/소다/스페셜': False,
    '과일주스/생과일': False,
}

all_items = []

menu_gallery = soup.find('ul', class_='menu_gallery tab_menu')
cat_lis = menu_gallery.find_all('li', recursive=False)

for li in cat_lis:
    # 카테고리명 추출 (첫 번째 텍스트)
    cat_name_full = li.get_text(strip=True)
    
    # 매칭되는 카테고리 찾기
    matched_cat = None
    for cat in AMASVIN_CATEGORIES:
        if cat_name_full.startswith(cat[:2]):
            matched_cat = cat
            break
    
    if not matched_cat:
        continue  # 베이커리 등 제외

    is_coffee = AMASVIN_CATEGORIES[matched_cat]
    print(f"\n▶ {matched_cat} 수집 시작")

    menu_items = li.find_all('li')
    for item in menu_items:
        h3 = item.find('h3')
        if not h3:
            continue
        name = h3.get_text(strip=True)
        all_items.append({
            'brand': '아마스빈',
            'name': name,
            'temp': None,
            'category': matched_cat,
            'is_coffee': is_coffee,
        })

    print(f"  {len([x for x in all_items if x['category'] == matched_cat])}개 수집")

df = pd.DataFrame(all_items)
df.to_csv('amasvin_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → amasvin_menu.csv")
print(df['category'].value_counts())


▶ 커피 수집 시작
  10개 수집

▶ 밀크티/라떼 수집 시작
  17개 수집

▶ 요거트/하동녹차/흑당 수집 시작
  13개 수집

▶ 쉐이크/스무디 수집 시작
  17개 수집

▶ 에이드/소다/스페셜 수집 시작
  13개 수집

▶ 과일주스/생과일 수집 시작
  11개 수집

✅ 완료! 총 81개 → amasvin_menu.csv
category
밀크티/라떼         17
쉐이크/스무디        17
요거트/하동녹차/흑당    13
에이드/소다/스페셜     13
과일주스/생과일       11
커피             10
Name: count, dtype: int64


## 18. 카페 봄봄

In [113]:
BOMBOM_CATEGORIES = {
    '신제품': ('menu_01.php', False),
    '커피': ('menu_02.php', True),
    '라떼': ('menu_03.php', False),
    '버블티': ('menu_04.php', False),
    '스무디': ('menu_05.php', False),
    '에이드': ('menu_06.php', False),
    '주스': ('menu_07.php', False),
    '티': ('menu_08.php', False),
    '저당/디카페인': ('menu_10.php', False),
}

BASE_URL = "https://cafebombom.co.kr/theme/nero/subpage/brand/"

all_items = []

for cat_name, (php, is_coffee) in BOMBOM_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    url = BASE_URL + php
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    items = soup.find_all('div', class_='item')
    for item in items:
        try:
            h2 = item.find('h2')
            if not h2:
                continue
            name = h2.get_text(strip=True)

            # temp 파싱 - span.cold 또는 span.hot
            c_h = item.find('div', class_='c_h')
            temp = None
            if c_h:
                if c_h.find('span', class_='cold') and c_h.find('span', class_='hot'):
                    temp = 'ICE/HOT'
                elif c_h.find('span', class_='cold'):
                    temp = 'ICE'
                elif c_h.find('span', class_='hot'):
                    temp = 'HOT'

            all_items.append({
                'brand': '카페봄봄',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")
    time.sleep(0.5)

df = pd.DataFrame(all_items)
df.to_csv('bombom_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → bombom_menu.csv")
print(df['category'].value_counts())


▶ 신제품 수집 시작
  12개 수집

▶ 커피 수집 시작
  19개 수집

▶ 라떼 수집 시작
  17개 수집

▶ 버블티 수집 시작
  4개 수집

▶ 스무디 수집 시작
  28개 수집

▶ 에이드 수집 시작
  9개 수집

▶ 주스 수집 시작
  11개 수집

▶ 티 수집 시작
  20개 수집

▶ 저당/디카페인 수집 시작
  10개 수집

✅ 완료! 총 130개 → bombom_menu.csv
category
스무디        28
티          20
커피         19
라떼         17
신제품        12
주스         11
저당/디카페인    10
에이드         9
버블티         4
Name: count, dtype: int64


## 19. 폴바셋

In [130]:
PAULBASSETT_CATEGORIES = {
    'COFFEE': ('A', True),
    'BEVERAGE': ('B', False),
}

all_items = []

driver = webdriver.Chrome(service=service, options=options)

# 1단계: dpid 수집
all_dpids = []
for cat_name, (cid1, is_coffee) in PAULBASSETT_CATEGORIES.items():
    print(f"\n▶ {cat_name} dpid 수집 시작")
    driver.get(f"https://www.baristapaulbassett.co.kr/menu/List.pb?cid1={cid1}")
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    for li in soup.find_all('li'):
        a = li.find('a', onclick=True)
        if a:
            match = re.search(r"goView\('(\w+)'\)", a['onclick'])
            if match:
                dpid = match.group(1)
                txt = li.find('div', class_='txtArea')
                if not txt:
                    continue
                full_text = txt.get_text(strip=True)
                name = re.sub(r'[A-Za-z].*', '', full_text).strip()
                if not name:
                    name = full_text
                all_dpids.append((dpid, name, cat_name, is_coffee, cid1))

    print(f"  {len([x for x in all_dpids if x[2] == cat_name])}개 dpid 수집")

print(f"\n총 {len(all_dpids)}개 dpid 수집 완료")

# 2단계: 상세 페이지에서 temp 수집
print("\n▶ 상세 페이지 temp 수집 시작")
for i, (dpid, name, cat_name, is_coffee, cid1) in enumerate(all_dpids):
    try:
        driver.get(f"https://www.baristapaulbassett.co.kr/menu/View.pb?cid1={cid1}&cid2=&dpid={dpid}")
        time.sleep(1.5)

        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # 구분(temp) 파싱
        temp = None
        for li in soup.find_all('li'):
            span = li.find('span')
            if span and '구분' in span.get_text():
                raw = li.get_text(strip=True).replace('구분', '').strip()
                if 'Hot' in raw and 'Ice' in raw:
                    temp = 'HOT/ICE'
                elif 'Hot' in raw:
                    temp = 'HOT'
                elif 'Ice' in raw or 'Iced' in raw:
                    temp = 'ICE'
                break

        all_items.append({
            'brand': '폴바셋',
            'name': name,
            'temp': temp,
            'category': cat_name,
            'is_coffee': is_coffee,
        })

        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(all_dpids)} 완료")

    except Exception as e:
        print(f"  에러 ({dpid}): {e}")
        continue

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('paulbassett_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → paulbassett_menu.csv")
print(df['category'].value_counts())


▶ COFFEE dpid 수집 시작
  63개 dpid 수집

▶ BEVERAGE dpid 수집 시작
  46개 dpid 수집

총 109개 dpid 수집 완료

▶ 상세 페이지 temp 수집 시작
  10/109 완료
  20/109 완료
  30/109 완료
  40/109 완료
  50/109 완료
  60/109 완료
  70/109 완료
  80/109 완료
  90/109 완료
  100/109 완료

✅ 완료! 총 109개 → paulbassett_menu.csv
category
COFFEE      63
BEVERAGE    46
Name: count, dtype: int64


## 20. 블루샥

In [11]:
BLUSHAAK_CATEGORIES = {
    'Coffee': ('menu01', True),
    'Beverage': ('menu02', False),
    'Blended': ('menu03', False),
    'Bottle': ('menu07', False),
    'New & Best': ('new_season', False),
}

BASE_URL = "https://www.blushaak.co.kr/bbs/board.php"

all_items = []

for cat_name, (bo_table, is_coffee) in BLUSHAAK_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    page = 1

    while True:
        url = f"{BASE_URL}?bo_table={bo_table}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        items = soup.find_all('div', class_='text_wrap')
        if not items:
            break

        for item in items:
            try:
                h3 = item.find('h3')
                if not h3:
                    continue
                name = h3.get_text(strip=True)

                temp = None
                dd = item.find('dd')
                if dd:
                    dd_text = dd.get_text(strip=True)
                    if '[ICE]' in dd_text and '[HOT]' in dd_text:
                        temp = 'HOT/ICE'
                    elif '[ICE]' in dd_text:
                        temp = 'ICE'
                    elif '[HOT]' in dd_text:
                        temp = 'HOT'

                all_items.append({
                    'brand': '블루샥',
                    'name': name,
                    'temp': temp,
                    'category': cat_name,
                    'is_coffee': is_coffee,
                })
            except Exception as e:
                print(f"  파싱 에러: {e}")
                continue

        # 다음 페이지 확인 - href에 page= 숫자로 확인
        next_page_link = soup.find('a', href=lambda h: h and f'page={page+1}' in h)
        if not next_page_link:
            break
        page += 1
        time.sleep(0.5)

    print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")
    time.sleep(0.5)

df = pd.DataFrame(all_items)
df.to_csv('blushaak_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → blushaak_menu.csv")
print(df['category'].value_counts())


▶ Coffee 수집 시작
  23개 수집

▶ Beverage 수집 시작
  29개 수집

▶ Blended 수집 시작
  25개 수집

▶ Bottle 수집 시작
  38개 수집

▶ New & Best 수집 시작
  0개 수집

✅ 완료! 총 115개 → blushaak_menu.csv
category
Bottle      38
Beverage    29
Blended     25
Coffee      23
Name: count, dtype: int64


## 21. 커피에 반하다

In [23]:
VANADA_CATEGORIES = {
    '에스프레소베이스': ('1', True),
    '밀크베이스': ('3', False),
    '에이드&주스': ('2', False),
    '티': ('6', False),
    '스무디&프라페': ('4', False),
}

BASE_URL = "https://vanada.kr/main/menu"

all_items = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, (page_type, is_coffee) in VANADA_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")

    # 목록 페이지에서 m_idx 수집
    list_url = f"{BASE_URL}/list.php?page_type={page_type}"
    driver.get(list_url)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    m_idxs = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        match = re.search(r'm_idx=(\d+)', href)
        if match:
            m_idx = match.group(1)
            # 메뉴명 추출
            p = a.find('p', class_='title')
            name = p.get_text(strip=True) if p else ''
            if m_idx not in [x[0] for x in m_idxs]:
                m_idxs.append((m_idx, name))

    print(f"  {len(m_idxs)}개 m_idx 수집")

    # 상세 페이지에서 temp 수집
    for m_idx, name in m_idxs:
        try:
            driver.get(f"{BASE_URL}/view.php?m_idx={m_idx}")
            time.sleep(1)

            soup = BeautifulSoup(driver.page_source, 'html.parser')

            # 테이블에서 ICED/HOT 확인
            tables = soup.find_all('table')
            temps = []
            for table in tables:
                th = table.find('th', class_='color')
                if th:
                    temps.append(th.get_text(strip=True))

            if 'ICED' in temps and 'HOT' in temps:
                temp = 'HOT/ICE'
            elif 'ICED' in temps:
                temp = 'ICE'
            elif 'HOT' in temps:
                temp = 'HOT'
            else:
                temp = None

            all_items.append({
                'brand': '커피에반하다',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
            })
        except Exception as e:
            print(f"  에러 ({m_idx}): {e}")
            continue

    print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('vanada_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → vanada_menu.csv")
print(df['category'].value_counts())


▶ 에스프레소베이스 수집 시작
  26개 m_idx 수집
  26개 수집

▶ 밀크베이스 수집 시작
  41개 m_idx 수집
  41개 수집

▶ 에이드&주스 수집 시작
  26개 m_idx 수집
  26개 수집

▶ 티 수집 시작
  38개 m_idx 수집
  38개 수집

▶ 스무디&프라페 수집 시작
  50개 m_idx 수집
  50개 수집

✅ 완료! 총 181개 → vanada_menu.csv
category
스무디&프라페     50
밀크베이스       41
티           38
에스프레소베이스    26
에이드&주스      26
Name: count, dtype: int64


## 22. 바나프레소

In [33]:
BANAPRESSO_CATEGORIES = {
    '커피': ('COFFEE', True),
    '저당&제로슈가': ('LOW & ZERO SUGAR', True),
    '디카페인커피': ('DECAFFEINE COFFEE', True),
    '빅바나': ('BIG BANA', True),
    '논커피라떼': ('NON COFFEE LATTE', False),
    '주스&드링크': ('JUICE & DRINK', False),
    '바나치노&스무디': ('BANACCINO & SMOOTHIE', False),
    '티&에이드': ('TEA & ADE', False),
}

# 1단계: 메뉴 목록 수집
driver = webdriver.Chrome(service=service, options=options)
driver.get("https://order.banapresso.com/#COFFEE")
time.sleep(3)

soup = BeautifulSoup(driver.page_source, 'html.parser')
h2_tags = soup.find_all('h2', class_='sc-gjSGGd')

menu_order = []
for h2 in h2_tags:
    span = h2.find('span')
    eng_name = span.get_text(strip=True) if span else ''

    matched_cat = None
    matched_is_coffee = False
    for cat_name, (eng, is_coffee) in BANAPRESSO_CATEGORIES.items():
        if eng == eng_name:
            matched_cat = cat_name
            matched_is_coffee = is_coffee
            break
    if not matched_cat:
        continue

    ul = h2.find_next('ul')
    if not ul:
        continue
    for li in ul.find_all('li', recursive=False):
        strong = li.find('strong')
        if strong:
            menu_order.append((strong.get_text(strip=True), matched_cat, matched_is_coffee))

print(f"총 {len(menu_order)}개 메뉴 수집")

# 2단계: 각 메뉴 인덱스로 클릭해서 temp 수집
all_items = []

for i, (name, cat_name, is_coffee) in enumerate(menu_order):
    try:
        # 매번 페이지 새로 로드
        driver.get("https://order.banapresso.com/#COFFEE")
        time.sleep(2)

        # 메뉴 아이템 다시 찾기
        menu_items = driver.find_elements(By.CSS_SELECTOR, 'li.sc-eeQa-dM')
        if i >= len(menu_items):
            print(f"  인덱스 초과: {i} ({name})")
            continue

        driver.execute_script("arguments[0].click();", menu_items[i])
        time.sleep(2)

        soup2 = BeautifulSoup(driver.page_source, 'html.parser')
        hot_btn = soup2.find('button', class_=lambda x: x and 'hot' in x)
        ice_btn = soup2.find('button', class_=lambda x: x and 'ice' in x)

        if hot_btn and ice_btn:
            temp = 'HOT/ICE'
        elif hot_btn:
            temp = 'HOT'
        elif ice_btn:
            temp = 'ICE'
        else:
            temp = None

        all_items.append({
            'brand': '바나프레소',
            'name': name,
            'temp': temp,
            'category': cat_name,
            'is_coffee': is_coffee,
        })

        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(menu_order)} 완료")

    except Exception as e:
        print(f"  에러 ({name}): {e}")
        continue

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('banapresso_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → banapresso_menu.csv")
print(df['category'].value_counts())

총 187개 메뉴 수집
  10/187 완료
  20/187 완료
  30/187 완료
  40/187 완료
  50/187 완료
  60/187 완료
  70/187 완료
  80/187 완료
  90/187 완료
  100/187 완료
  110/187 완료
  120/187 완료
  130/187 완료
  140/187 완료
  150/187 완료
  160/187 완료
  170/187 완료
  180/187 완료

✅ 완료! 총 187개 → banapresso_menu.csv
category
티&에이드       34
커피          30
빅바나         26
논커피라떼       26
바나치노&스무디    25
디카페인커피      23
저당&제로슈가     13
주스&드링크      10
Name: count, dtype: int64


## 23. 하이오커피

In [48]:
HIO_CATEGORIES = {
    'NEW': ('NEW', False),
    'COFFEE': ('COFFEE', True),
    'COLD BREW': ('COLD+BREW', True),
    'BEVERAGE': ('BEVERAGE', False),
    'BLENDED': ('BLENDED', False),
    'ADE&TEA': ('ADE%EF%BC%86TEA', False),
    "BUBBLIN'": ('BUBBLIN%E2%80%B2', False),
    'SIGNATURE': ('SIGNATURE', False),
    '1L BOTTLE': ('1L+BOTTLE', False),
}

BASE_URL = "https://hiocoffee.com/bbs/board.php?bo_table=03_02"
all_items = []

for cat_name, (sca, is_coffee) in HIO_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    page = 1

    while True:
        url = f"{BASE_URL}&sca={sca}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        items = soup.find_all('ul', class_='gallery-list')
        all_li = []
        for ul in items:
            all_li.extend(ul.find_all('li'))

        if not all_li:
            break

        for li in all_li:
            try:
                name_tag = li.find('p', class_='title')
                if not name_tag:
                    continue
                name = name_tag.get_text(strip=True)

                # temp 파싱
                temp = None
                temp_tag = li.find('span', class_='ice')
                if temp_tag:
                    temp_text = temp_tag.get_text(strip=True).upper()
                    if 'HOT' in temp_text and 'ICE' in temp_text:
                        temp = 'HOT/ICE'
                    elif 'HOT' in temp_text:
                        temp = 'HOT'
                    elif 'ICE' in temp_text:
                        temp = 'ICE'
                    else:
                        temp = ''
                else:
                    temp = ''

                all_items.append({
                    'brand': '하이오커피',
                    'name': name,
                    'temp': temp,
                    'category': cat_name,
                    'is_coffee': is_coffee,
                })
            except Exception as e:
                print(f"  파싱 에러: {e}")
                continue

        # 다음 페이지 확인
        next_page = soup.find('a', href=lambda h: h and f'page={page+1}' in h)
        if not next_page:
            break
        page += 1
        time.sleep(0.5)

    print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")
    time.sleep(0.5)

df = pd.DataFrame(all_items)
df.to_csv('hiocoffee_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → hiocoffee_menu.csv")
print(df['category'].value_counts())


▶ NEW 수집 시작
  31개 수집

▶ COFFEE 수집 시작
  22개 수집

▶ COLD BREW 수집 시작
  10개 수집

▶ BEVERAGE 수집 시작
  15개 수집

▶ BLENDED 수집 시작
  17개 수집

▶ ADE&TEA 수집 시작
  25개 수집

▶ BUBBLIN' 수집 시작
  3개 수집

▶ SIGNATURE 수집 시작
  5개 수집

▶ 1L BOTTLE 수집 시작
  32개 수집

✅ 완료! 총 160개 → hiocoffee_menu.csv
category
1L BOTTLE    32
NEW          31
ADE&TEA      25
COFFEE       22
BLENDED      17
BEVERAGE     15
COLD BREW    10
SIGNATURE     5
BUBBLIN'      3
Name: count, dtype: int64


## 24. 디저트39

In [68]:
DESSERT39_CATEGORIES = {
    'SEASON&NEW': ('1', '137', False),
    'BEST': ('4', '106', False),
    'ZERO&LOW calorie': ('6', '111', False),
    'REUSABLE CUP': ('7', '105', False),
    'NO SUGAR': ('8', '123', False),
    'COFFEE': ('9', '98', True),
    'NON-COFFEE': ('10', '100', False),
    'TEA BLENDED&ADE': ('13', '121', False),
}

all_items = []
driver = webdriver.Chrome(service=service, options=options)
driver.get("https://dessert39.com/html/pages/menu_beverage.php")
time.sleep(3)

for cat_name, (cont_num, tab_idx, is_coffee) in DESSERT39_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    cont_class = f'cont-wrap{cont_num}'

    try:
        # ICE 탭 클릭
        ice_tab = driver.find_element(By.CSS_SELECTOR, f'li[data-idx="{tab_idx}_ice"]')
        driver.execute_script("arguments[0].click();", ice_tab)
        time.sleep(1)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        cont = soup.find('div', class_=cont_class)
        ice_names = set([p.find('p', class_='tit').get_text(strip=True) for p in cont.find_all('div', class_='product') if p.find('p', class_='tit')]) if cont else set()

        # HOT 탭 클릭
        hot_tab = driver.find_element(By.CSS_SELECTOR, f'li[data-idx="{tab_idx}_hot"]')
        driver.execute_script("arguments[0].click();", hot_tab)
        time.sleep(1)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        cont = soup.find('div', class_=cont_class)
        hot_names = set([p.find('p', class_='tit').get_text(strip=True) for p in cont.find_all('div', class_='product') if p.find('p', class_='tit')]) if cont else set()

        # ALL 탭 클릭
        all_tab = driver.find_element(By.CSS_SELECTOR, f'li[data-idx="{tab_idx}_"]')
        driver.execute_script("arguments[0].click();", all_tab)
        time.sleep(1)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        cont = soup.find('div', class_=cont_class)
        if not cont:
            print(f"  섹션 없음")
            continue

        for p in cont.find_all('div', class_='product'):
            tit = p.find('p', class_='tit')
            if not tit:
                continue
            name = tit.get_text(strip=True)

            if name in ice_names and name in hot_names:
                temp = 'HOT/ICE'
            elif name in ice_names:
                temp = 'ICE'
            elif name in hot_names:
                temp = 'HOT'
            else:
                temp = None

            all_items.append({
                'brand': '디저트39',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
            })

        print(f"  {len([x for x in all_items if x['category'] == cat_name])}개 수집")

    except Exception as e:
        print(f"  에러: {e}")
        continue

driver.quit()

df = pd.DataFrame(all_items)
df.to_csv('dessert39_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items)}개 → dessert39_menu.csv")
print(df['category'].value_counts())


▶ SEASON&NEW 수집 시작
  35개 수집

▶ BEST 수집 시작
  32개 수집

▶ ZERO&LOW calorie 수집 시작
  40개 수집

▶ REUSABLE CUP 수집 시작
  15개 수집

▶ NO SUGAR 수집 시작
  15개 수집

▶ COFFEE 수집 시작
  35개 수집

▶ NON-COFFEE 수집 시작
  69개 수집

▶ TEA BLENDED&ADE 수집 시작
  13개 수집

✅ 완료! 총 254개 → dessert39_menu.csv
category
NON-COFFEE          69
ZERO&LOW calorie    40
SEASON&NEW          35
COFFEE              35
BEST                32
REUSABLE CUP        15
NO SUGAR            15
TEA BLENDED&ADE     13
Name: count, dtype: int64


In [85]:
driver = webdriver.Chrome(service=service, options=options)
driver.get("https://www.starbucks.co.kr/menu/drink_list.do")
time.sleep(4)

soup = BeautifulSoup(driver.page_source, 'html.parser')
product_list = soup.find('div', class_='product_list')

all_items_starbucks = []
current_category = ''

COFFEE_CATEGORIES = {'콜드 브루 커피', '브루드 커피', '에스프레소'}

for dl in product_list.find_all('dl'):
    try:
        # 카테고리 헤더인지 확인
        dt = dl.find('dt')
        if dt and dt.find('a', class_=lambda x: not x or 'goDrinkView' not in str(x)):
            a = dt.find('a')
            if a and 'goDrinkView' not in str(a.get('class', '')):
                cat_text = a.get_text(strip=True)
                if cat_text:
                    current_category = cat_text
                    continue

        # 메뉴 아이템
        img = dl.find('img', alt=True)
        if not img:
            continue
        name = img.get('alt', '').strip()
        if not name or name in ['NEW', 'BEST', '']:
            continue

        # temp 추출
        if name.startswith('아이스') or name.upper().startswith('ICE'):
            temp = 'ICE'
        else:
            temp = 'HOT'

        all_items_starbucks.append({
            'brand': '스타벅스',
            'name': name,
            'temp': temp,
            'category': current_category,
            'is_coffee': current_category in COFFEE_CATEGORIES,
            'price': None
        })
    except Exception as e:
        print(f"파싱 에러: {e}")
        continue

driver.quit()

df = pd.DataFrame(all_items_starbucks)
df.to_csv('starbucks_menu.csv', index=False, encoding='utf-8-sig')
print(f"✅ 완료! 총 {len(all_items_starbucks)}개 → starbucks_menu.csv")
print(df['category'].value_counts())
print(df.head(10))

✅ 완료! 총 196개 → starbucks_menu.csv
category
콜드 브루 커피    196
Name: count, dtype: int64
  brand                   name temp  category  is_coffee price
0  스타벅스  바삭 피스타치오 바닐라 크림 콜드 브루  HOT  콜드 브루 커피       True  None
1  스타벅스           서울 막걸리향 콜드브루  HOT  콜드 브루 커피       True  None
2  스타벅스            나이트로 바닐라 크림  HOT  콜드 브루 커피       True  None
3  스타벅스             나이트로 콜드 브루  HOT  콜드 브루 커피       True  None
4  스타벅스               돌체 콜드 브루  HOT  콜드 브루 커피       True  None
5  스타벅스               리저브 나이트로  HOT  콜드 브루 커피       True  None
6  스타벅스              리저브 콜드 브루  HOT  콜드 브루 커피       True  None
7  스타벅스          막걸리향 크림 콜드 브루  HOT  콜드 브루 커피       True  None
8  스타벅스               민트 콜드 브루  HOT  콜드 브루 커피       True  None
9  스타벅스           바닐라 크림 콜드 브루  HOT  콜드 브루 커피       True  None


In [87]:
STARBUCKS_CATEGORIES = {
    '콜드 브루 커피': ('product_cold_brew', True),
    '브루드 커피': ('product_brewed', True),
    '에스프레소': ('product_espresso', True),
    '프라푸치노': ('product_frappuccino', False),
    '블렌디드': ('product_blended', False),
    '스타벅스 리프레셔': ('product_refresher', False),
    '스타벅스 피지오': ('product_fizzio', False),
    '티(티바나)': ('product_tea', False),
    '기타 제조 음료': ('product_etc', False),
}

all_items_starbucks = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, (input_id, is_coffee) in STARBUCKS_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://www.starbucks.co.kr/menu/drink_list.do")
    time.sleep(4)

    # 전체 해제 후 카테고리 선택을 JS로 직접 처리
    driver.execute_script("""
        // 전체 체크박스 해제
        var allCb = document.getElementById('product_all');
        if(allCb.checked) allCb.click();
    """)
    time.sleep(1)

    driver.execute_script(f"""
        var cb = document.getElementById('{input_id}');
        cb.click();
    """)
    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    product_list = soup.find('div', class_='product_list')
    if not product_list:
        print("  product_list 없음")
        continue

    # 보이는 dl만 수집 (display:none 제외)
    for dl in product_list.find_all('dl'):
        try:
            dd = dl.find('dd')
            if not dd:
                continue
            # display:none인 dd 제외
            style = dd.get('style', '')
            if 'none' in style:
                continue

            img = dl.find('img', alt=True)
            if not img:
                continue
            name = img.get('alt', '').strip()
            if not name or name in ['NEW', 'BEST', 'DECAF', '']:
                continue

            if name.startswith('아이스') or name.upper().startswith('ICE'):
                temp = 'ICE'
            else:
                temp = 'HOT'

            all_items_starbucks.append({
                'brand': '스타벅스',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    count = len([x for x in all_items_starbucks if x['category'] == cat_name])
    print(f"  {count}개 수집")

driver.quit()

df = pd.DataFrame(all_items_starbucks)
df.to_csv('starbucks_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_starbucks)}개 → starbucks_menu.csv")
print(df['category'].value_counts())


▶ 콜드 브루 커피 수집 시작
  196개 수집

▶ 브루드 커피 수집 시작
  196개 수집

▶ 에스프레소 수집 시작
  196개 수집

▶ 프라푸치노 수집 시작
  196개 수집

▶ 블렌디드 수집 시작
  196개 수집

▶ 스타벅스 리프레셔 수집 시작
  196개 수집

▶ 스타벅스 피지오 수집 시작
  196개 수집

▶ 티(티바나) 수집 시작
  196개 수집

▶ 기타 제조 음료 수집 시작
  196개 수집

✅ 완료! 총 1764개 → starbucks_menu.csv
category
콜드 브루 커피     196
브루드 커피       196
에스프레소        196
프라푸치노        196
블렌디드         196
스타벅스 리프레셔    196
스타벅스 피지오     196
티(티바나)       196
기타 제조 음료     196
Name: count, dtype: int64


In [100]:
DALCU_CATEGORIES = {
    'Coffee': ('step_01', True),
    'Non coffee': ('step_02', False),
    'Juice': ('step_03', False),
    'Smoothie & Frappe': ('step_04', False),
    'Ade & tea': ('step_05', False),
}

all_items_dalcu = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, (step, is_coffee) in DALCU_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://dalcu.co.kr/bbs/content.php?co_id=menu&step=step_02")
    time.sleep(3)

    # 카테고리 탭 클릭
    try:
        tab = driver.find_element(By.CSS_SELECTOR, f'div.menu_button_s_02[data-value="{step}"]')
        driver.execute_script("arguments[0].click();", tab)
        time.sleep(2)
    except Exception as e:
        print(f"  탭 클릭 에러: {e}")
        continue

    while True:
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        items = soup.find_all('div', class_='menu_border_s_text_div')

        for item in items:
            try:
                name = item.get_text(strip=True)
                if not name:
                    continue

                # temp 추출
                if '(ICE)' in name:
                    temp = 'ICE'
                elif '(HOT)' in name:
                    temp = 'HOT'
                else:
                    temp = ''

                # temp 괄호 제거
                clean_name = name.replace('(ICE)', '').replace('(HOT)', '').strip()

                all_items_dalcu.append({
                    'brand': '달리는커피',
                    'name': clean_name,
                    'temp': temp,
                    'category': cat_name,
                    'is_coffee': is_coffee,
                    'price': None
                })
            except Exception as e:
                print(f"  파싱 에러: {e}")
                continue

        # 다음 페이지 버튼 확인
        try:
            next_btn = driver.find_element(By.CSS_SELECTOR, 'div.swiper-button-next')
            if 'swiper-button-disabled' in next_btn.get_attribute('class'):
                break
            driver.execute_script("arguments[0].click();", next_btn)
            time.sleep(1)
        except:
            break

    count = len([x for x in all_items_dalcu if x['category'] == cat_name])
    print(f"  {count}개 수집")

driver.quit()

df = pd.DataFrame(all_items_dalcu)
df.to_csv('dalcu_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_dalcu)}개 → dalcu_menu.csv")
print(df['category'].value_counts())


▶ Coffee 수집 시작
  209개 수집

▶ Non coffee 수집 시작
  209개 수집

▶ Juice 수집 시작
  209개 수집

▶ Smoothie & Frappe 수집 시작
  209개 수집

▶ Ade & tea 수집 시작
  209개 수집

✅ 완료! 총 1045개 → dalcu_menu.csv
category
Coffee               209
Non coffee           209
Juice                209
Smoothie & Frappe    209
Ade & tea            209
Name: count, dtype: int64


## 25. 파스쿠찌

In [119]:
PASCUCCI_CATEGORIES = {
    '이탈리안커피': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00100010', True),
    '커피(HOT)': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00100020', True),
    '커피(ICED)': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00100030', True),
    '콜드브루': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00100040', True),
    '시즌음료': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00200010', False),
    '그라니따': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00200020', False),
    '티': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00200030', False),
    '기타음료': ('https://www.pascucci.co.kr/product/productList.asp?typeCode=00200050', False),
}

all_items_pascucci = []

for cat_name, (url, is_coffee) in PASCUCCI_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    product_wrap = soup.find('ul', class_='productWrap')
    if not product_wrap:
        print("  메뉴 없음")
        continue

    for a in product_wrap.find_all('a', class_='product'):
        try:
            name_tag = a.find('h2')
            if not name_tag:
                continue
            name = name_tag.get_text(strip=True)

            eng_tag = a.find('p', class_='titleEng')
            eng_name = eng_tag.get_text(strip=True) if eng_tag else ''

            # temp 추출
            if '(ICED)' in cat_name or 'ICE' in name.upper() or '아이스' in name:
                temp = 'ICE'
            elif '(HOT)' in cat_name or 'HOT' in name.upper():
                temp = 'HOT'
            else:
                temp = ''

            all_items_pascucci.append({
                'brand': '파스쿠찌',
                'name': name,
                'name_en': eng_name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    print(f"  {len([x for x in all_items_pascucci if x['category'] == cat_name])}개 수집")
    time.sleep(1)

df = pd.DataFrame(all_items_pascucci)
df.to_csv('pascucci_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_pascucci)}개 → pascucci_menu.csv")
print(df['category'].value_counts())


▶ 이탈리안커피 수집 시작
  5개 수집

▶ 커피(HOT) 수집 시작
  8개 수집

▶ 커피(ICED) 수집 시작
  10개 수집

▶ 콜드브루 수집 시작
  4개 수집

▶ 시즌음료 수집 시작
  14개 수집

▶ 그라니따 수집 시작
  9개 수집

▶ 티 수집 시작
  12개 수집

▶ 기타음료 수집 시작
  21개 수집

✅ 완료! 총 83개 → pascucci_menu.csv
category
기타음료        21
시즌음료        14
티           12
커피(ICED)    10
그라니따         9
커피(HOT)      8
이탈리안커피       5
콜드브루         4
Name: count, dtype: int64


## 26. 더리터 커피

In [128]:
LITER_TAB_IDS = {
    '커피': ('s202503202bbc29b7cac08', True),
    '밀크베이스': ('s202503209557e9a1c5f58', False),
    '요거스&버블티': ('s20250320c1c5c166e253d', False),
    '프라페': ('s2025032068364581d97e3', False),
    '에이드&주스': ('s20250320c1f9d8d8f3ce6', False),
    '티': ('s20250320196feb020357b', False),
}

all_items_liter = []

for cat_name, (tab_id, is_coffee) in LITER_TAB_IDS.items():
    section = soup.find(id=tab_id)
    if not section:
        print(f"{cat_name}: 섹션 못찾음")
        continue

    for item in section.find_all('p', class_='title'):
        try:
            name = item.get_text(strip=True)
            if not name:
                continue

            all_items_liter.append({
                'brand': '더리터',
                'name': name,
                'temp': '',
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"파싱 에러: {e}")
            continue

    print(f"▶ {cat_name}: {len([x for x in all_items_liter if x['category'] == cat_name])}개")

df = pd.DataFrame(all_items_liter)
df.to_csv('theliter_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_liter)}개 → theliter_menu.csv")
print(df['category'].value_counts())

▶ 커피: 17개
▶ 밀크베이스: 13개
▶ 요거스&버블티: 11개
▶ 프라페: 12개
▶ 에이드&주스: 13개
▶ 티: 16개

✅ 완료! 총 82개 → theliter_menu.csv
category
커피         17
티          16
밀크베이스      13
에이드&주스     13
프라페        12
요거스&버블티    11
Name: count, dtype: int64


## 27. 카페인 중독

In [135]:
CAFFEINE_CATEGORIES = {
    'SIGNATURE': ('https://xn--iq1bo78ac9at1k9mh.com/65', False),
    'COFFEE & NON-COFFEE': ('https://xn--iq1bo78ac9at1k9mh.com/979821801', True),
    'TEA': ('https://xn--iq1bo78ac9at1k9mh.com/1373078117', False),
    'ADE': ('https://xn--iq1bo78ac9at1k9mh.com/40', False),
    'SMOOTHIE': ('https://xn--iq1bo78ac9at1k9mh.com/41', False),
    '제주 한정': ('https://xn--iq1bo78ac9at1k9mh.com/70', False),
}

all_items_caffeine = []

for cat_name, (url, is_coffee) in CAFFEINE_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    for div in soup.find_all('div', id=lambda x: x and x.startswith('caption_')):
        try:
            h4 = div.find('h4')
            if not h4:
                continue
            name = h4.get_text(strip=True)
            if not name:
                continue

            all_items_caffeine.append({
                'brand': '카페인중독',
                'name': name,
                'temp': '',
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    count = len([x for x in all_items_caffeine if x['category'] == cat_name])
    print(f"  {count}개 수집")
    time.sleep(1)

df = pd.DataFrame(all_items_caffeine)
df.to_csv('caffeine_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_caffeine)}개 → caffeine_menu.csv")
print(df['category'].value_counts())


▶ SIGNATURE 수집 시작
  2개 수집

▶ COFFEE & NON-COFFEE 수집 시작
  25개 수집

▶ TEA 수집 시작
  11개 수집

▶ ADE 수집 시작
  7개 수집

▶ SMOOTHIE 수집 시작
  13개 수집

▶ 제주 한정 수집 시작
  4개 수집

✅ 완료! 총 62개 → caffeine_menu.csv
category
COFFEE & NON-COFFEE    25
SMOOTHIE               13
TEA                    11
ADE                     7
제주 한정                   4
SIGNATURE               2
Name: count, dtype: int64


## 28. 우지커피

In [143]:
OOZY_CATEGORIES = {
    'COFFEE': ('https://oozycoffee.com/Technology', True),
    'COLD BREW': ('https://oozycoffee.com/27', True),
    'BEVERAGE': ('https://oozycoffee.com/28', False),
    'FRAPPE': ('https://oozycoffee.com/29', False),
    'ADE & MOJITO': ('https://oozycoffee.com/30', False),
    'TEA & JUICE': ('https://oozycoffee.com/31', False),
}

all_items_oozy = []

for cat_name, (url, is_coffee) in OOZY_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')

    for div in soup.find_all('div', id=lambda x: x and x.startswith('caption_')):
        try:
            h4 = div.find('h4')
            if not h4:
                continue
            name = h4.get_text(strip=True)
            if not name:
                continue

            all_items_oozy.append({
                'brand': '우지커피',
                'name': name,
                'temp': '',
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    count = len([x for x in all_items_oozy if x['category'] == cat_name])
    print(f"  {count}개 수집")
    time.sleep(1)

df = pd.DataFrame(all_items_oozy)
df.to_csv('oozy_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_oozy)}개 → oozy_menu.csv")
print(df['category'].value_counts())


▶ COFFEE 수집 시작
  80개 수집

▶ COLD BREW 수집 시작
  28개 수집

▶ BEVERAGE 수집 시작
  56개 수집

▶ FRAPPE 수집 시작
  98개 수집

▶ ADE & MOJITO 수집 시작
  26개 수집

▶ TEA & JUICE 수집 시작
  58개 수집

✅ 완료! 총 346개 → oozy_menu.csv
category
FRAPPE          98
COFFEE          80
TEA & JUICE     58
BEVERAGE        56
COLD BREW       28
ADE & MOJITO    26
Name: count, dtype: int64


## 29. 백억커피

In [157]:
BAEKUK_CATEGORIES = {
    '시그니처': ('ca0', False),
    '커피': ('ca1', True),
    '콜드브루': ('ca2', True),
    '라떼': ('ca3', False),
    '찐우유': ('ca4', False),
    '버블티': ('ca5', False),
    '요거트스무디': ('ca6', False),
    '스무디': ('ca7', False),
    '에이드': ('ca8', False),
    '주스': ('ca9', False),
    '티': ('ca10', False),
}

all_items_baekuk = []
driver = webdriver.Chrome(service=service, options=options)

for cat_name, (cb_id, is_coffee) in BAEKUK_CATEGORIES.items():
    print(f"\n▶ {cat_name} 수집 시작")
    driver.get("https://10billioncoffee.co.kr/")
    time.sleep(3)

    # 체크박스 label 클릭
    try:
        label = driver.find_element(By.CSS_SELECTOR, f'label[for="{cb_id}"]')
        driver.execute_script("arguments[0].click();", label)
        time.sleep(2)
    except Exception as e:
        print(f"  체크박스 클릭 에러: {e}")
        continue

    # MORE+ 버튼 계속 클릭
    while True:
        try:
            more_btn = driver.find_element(By.CSS_SELECTOR, 'button.more')
            if more_btn.is_displayed():
                driver.execute_script("arguments[0].click();", more_btn)
                time.sleep(1)
            else:
                break
        except:
            break

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    menu_list = soup.find('div', class_='menu_list')
    if not menu_list:
        print("  메뉴 없음")
        continue

    for li in menu_list.find_all('li'):
        try:
            p = li.find('p')
            if not p:
                continue
            name = p.get_text(strip=True)
            if not name:
                continue

            # temp 확인 (span 태그에서)
            temp_tag = li.find('span')
            temp_text = temp_tag.get_text(strip=True).upper() if temp_tag else ''
            if 'HOT' in temp_text and 'ICE' in temp_text:
                temp = 'HOT/ICE'
            elif 'HOT' in temp_text:
                temp = 'HOT'
            elif 'ICE' in temp_text:
                temp = 'ICE'
            else:
                temp = ''

            all_items_baekuk.append({
                'brand': '백억커피',
                'name': name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

    count = len([x for x in all_items_baekuk if x['category'] == cat_name])
    print(f"  {count}개 수집")

driver.quit()

df = pd.DataFrame(all_items_baekuk)
df.to_csv('baekuk_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_baekuk)}개 → baekuk_menu.csv")
print(df['category'].value_counts())


▶ 시그니처 수집 시작
  4개 수집

▶ 커피 수집 시작
  11개 수집

▶ 콜드브루 수집 시작
  3개 수집

▶ 라떼 수집 시작
  11개 수집

▶ 찐우유 수집 시작
  5개 수집

▶ 버블티 수집 시작
  3개 수집

▶ 요거트스무디 수집 시작
  4개 수집

▶ 스무디 수집 시작
  9개 수집

▶ 에이드 수집 시작
  8개 수집

▶ 주스 수집 시작
  4개 수집

▶ 티 수집 시작
  13개 수집

✅ 완료! 총 75개 → baekuk_menu.csv
category
티         13
커피        11
라떼        11
스무디        9
에이드        8
찐우유        5
시그니처       4
요거트스무디     4
주스         4
콜드브루       3
버블티        3
Name: count, dtype: int64


## 30. 달리는 커피-보류

In [164]:
DALCU_CATEGORIES = [
    ('Coffee', True),
    ('Non coffee', False),
    ('Juice', False),
    ('Smoothie & Frappe', False),
    ('Ade & tea', False),
]

all_items_dalcu = []
driver = webdriver.Chrome(service=service, options=options)
driver.get("https://dalcu.co.kr/bbs/content.php?co_id=menu")
time.sleep(3)

# 음료 탭 클릭
menu_tab = driver.find_element(By.XPATH, "//div[contains(@class,'menu_button_s') and .//p[text()='음료']]")
driver.execute_script("arguments[0].click();", menu_tab)
time.sleep(2)

# 화살표 버튼 계속 클릭해서 모든 메뉴 로딩
for _ in range(20):
    try:
        next_btns = driver.find_elements(By.CSS_SELECTOR, 'div.next_page')
        if next_btns:
            driver.execute_script("arguments[0].click();", next_btns[0])
            time.sleep(0.5)
        else:
            break
    except:
        break

soup = BeautifulSoup(driver.page_source, 'html.parser')
groups = soup.find_all('div', class_='menu_border_02')

for i, (cat_name, is_coffee) in enumerate(DALCU_CATEGORIES):
    if i >= len(groups):
        break
    group = groups[i]
    items = group.find_all('div', class_='menu_border_s_text_div')
    print(f"\n▶ {cat_name}: {len(items)}개")

    for item in items:
        try:
            name = item.get_text(strip=True)
            if not name or name == '달리블':
                continue

            # temp 추출
            if '(ICE)' in name:
                temp = 'ICE'
            elif '(HOT)' in name:
                temp = 'HOT'
            else:
                temp = ''

            # temp 괄호 제거
            clean_name = name.replace('(ICE)', '').replace('(HOT)', '').strip()

            all_items_dalcu.append({
                'brand': '달리는커피',
                'name': clean_name,
                'temp': temp,
                'category': cat_name,
                'is_coffee': is_coffee,
                'price': None
            })
        except Exception as e:
            print(f"  파싱 에러: {e}")
            continue

driver.quit()

df = pd.DataFrame(all_items_dalcu)
df.to_csv('dalcu_menu.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! 총 {len(all_items_dalcu)}개 → dalcu_menu.csv")
print(df['category'].value_counts())


▶ Coffee: 21개

▶ Non coffee: 17개

▶ Juice: 9개

▶ Smoothie & Frappe: 14개

▶ Ade & tea: 32개

✅ 완료! 총 88개 → dalcu_menu.csv
category
Ade & tea            31
Coffee               20
Non coffee           16
Smoothie & Frappe    13
Juice                 8
Name: count, dtype: int64
